# TinyDoc-VLM: Colab Training

Fully automatic. Run all cells in order. Disconnect-safe — re-run from the top to resume.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/tinydoc-vlm'
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
print(f'Drive ready at {DRIVE_ROOT}')

## 2. Clone repo & install deps

In [ ]:
import os, sys, subprocess, requests, zipfile, io, shutil

REPO_DIR = '/content/tinydoc-vlm'

# Use zip download — avoids git CWD issues on Colab reconnect
if not os.path.exists(f'{REPO_DIR}/data/synthetic/generator.py'):
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    print('Downloading repo via zip...')
    r = requests.get('https://github.com/eulogik/TinyDoc-VLM/archive/refs/heads/main.zip', timeout=60)
    z = zipfile.ZipFile(io.BytesIO(r.content))
    z.extractall('/content')
    os.rename('/content/TinyDoc-VLM-main', REPO_DIR)

assert os.path.exists(f'{REPO_DIR}/data/synthetic/generator.py'), 'Download failed'

os.chdir(REPO_DIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torchvision', '--index-url',
    'https://download.pytorch.org/whl/cu121'], cwd=REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers', 'sentencepiece', 'tokenizers', 'pillow', 'numpy', 'pandas',
    'tqdm', 'pyyaml', 'gradio', 'optimum', 'onnx', 'onnxruntime', 'einops',
    'faker', 'jinja2', 'pydantic', 'accelerate'], cwd=REPO_DIR)
print('Repo ready — deps installed')


## 3. Generate synthetic data

Generates on Colab's local SSD (fast). Verifies every image exists after generation.

In [ ]:
import os, sys, subprocess, json, shutil
from pathlib import Path

DATA_DIR = Path('/content/tinydoc-vlm/data/synthetic')
MANIFEST = DATA_DIR / 'output' / 'manifest.jsonl'
REPO = '/content/tinydoc-vlm'

def verify_data():
    if not MANIFEST.exists():
        return False
    with open(MANIFEST) as f:
        entries = [line.strip() for line in f if line.strip()]
    if len(entries) < 900:
        return False
    for line in entries:
        img_path = DATA_DIR / json.loads(line)['image_path']
        if not img_path.exists():
            return False
    return True

if verify_data():
    print(f'Data OK - {sum(1 for _ in open(MANIFEST))} docs')
else:
    print('Generating 1000 synthetic documents (~5 min)...')
    if DATA_DIR.exists() and (DATA_DIR / 'output').exists():
        shutil.rmtree(str(DATA_DIR / 'output'))
    result = subprocess.run([
        sys.executable, '-c',
        'import sys; sys.path.insert(0, "' + REPO + '"); '
        'from data.synthetic.generator import generate_synthetic_documents; '
        'generate_synthetic_documents(num_docs=1000, output_dir="' + str(DATA_DIR / 'output') + '")'
    ], cwd=REPO, capture_output=True, text=True, timeout=600)
    print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
    if result.returncode != 0:
        print(result.stderr[-2000:])
    assert verify_data(), 'Data generation failed'
    print('Done!')


## 4. Train all 3 stages

Saves to local SSD. Syncs final checkpoint to Drive at each stage end (overwrites, no duplicates).

In [ ]:
import yaml
import torch
import shutil
import logging
from torch.utils.data import random_split
from pathlib import Path

from tinydoc_vlm import (
    TinyDocVLMConfig,
    TinyDocVLMForConditionalGeneration,
    TinyDocVLMProcessor,
    TinyDocVLMTrainer,
    TrainerConfig,
    DocumentDataset,
)

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger('colab')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f'Device: {device}')
assert device.type == 'cuda', 'T4 GPU not detected! Runtime → Change runtime type → T4 GPU → Restart'

# Remove any old step/init cruft from Drive (free space)
for d in Path(f'{DRIVE_ROOT}/checkpoints').rglob('step_*'):
    shutil.rmtree(d, ignore_errors=True)
for d in Path(f'{DRIVE_ROOT}/checkpoints').rglob('init'):
    shutil.rmtree(d, ignore_errors=True)

STAGE_CONFIGS = {
    1: '/content/tinydoc-vlm/training/stage1_layout_pretrain.yaml',
    2: '/content/tinydoc-vlm/training/stage2_doc_understanding.yaml',
    3: '/content/tinydoc-vlm/training/stage3_instruction_tuning.yaml',
}

def build_config(stage):
    with open(STAGE_CONFIGS[stage]) as f:
        cfg = yaml.safe_load(f)
    cfg['training']['batch_size'] = 4
    cfg['training']['gradient_accumulation_steps'] = 8
    cfg['training']['save_every_steps'] = 999999
    cfg['training']['eval_every_steps'] = 250
    cfg['training']['log_every_steps'] = 10
    cfg['training']['gradient_checkpointing'] = True
    cfg['training']['num_epochs'] = 1
    cfg['training']['learning_rate'] = float(cfg['training']['learning_rate'])
    cfg['training']['output_dir'] = f'/content/checkpoints/stage{stage}'
    cfg['data']['manifest_path'] = '/content/tinydoc-vlm/data/synthetic/output/manifest.jsonl'
    cfg['data']['stage'] = stage
    if stage > 1:
        prev_local = Path(f'/content/checkpoints/stage{stage-1}')
        epochs = sorted(prev_local.glob('epoch_*'))
        if not epochs:
            prev_drive = Path(f'{DRIVE_ROOT}/checkpoints/stage{stage-1}')
            epochs = sorted(prev_drive.glob('epoch_*'))
        if epochs:
            cfg['model']['pretrained_checkpoint'] = str(epochs[-1])
            logger.info(f'Loading stage {stage-1} checkpoint from: {epochs[-1]}')
        else:
            logger.warning(f'No stage {stage-1} checkpoint found - starting from scratch')
    return cfg

def run_stage(stage):
    logger.info(f'\n===== STAGE {stage} =====')
    cfg = build_config(stage)
    tc = cfg['training']
    local_dir = Path(tc['output_dir'])
    drive_dir = Path(f'{DRIVE_ROOT}/checkpoints/stage{stage}')
    drive_done = (drive_dir / 'stage_complete.txt').exists()

    if drive_done:
        logger.info(f'Stage {stage} already complete on Drive. Skipping.')
        return

    mc = cfg['model']
    config = TinyDocVLMConfig(
        vision_config=mc.get('vision_config'),
        decoder_config=mc.get('decoder_config'),
        pixel_shuffle_scale=mc.get('pixel_shuffle_scale', 3),
        image_size=mc.get('image_size', 384),
        patch_size=mc.get('patch_size', 16),
    )
    model = TinyDocVLMForConditionalGeneration(config)

    def load_model_weights(path):
        state_path = Path(path) / 'model_state.pt'
        if state_path.exists():
            m = TinyDocVLMForConditionalGeneration(config)
            m.load_state_dict(torch.load(str(state_path), map_location='cpu', weights_only=True))
            logger.info(f'Loaded weights from {state_path}')
            return m
        return TinyDocVLMForConditionalGeneration.from_pretrained(path)

    pretrained = mc.get('pretrained_checkpoint')
    if pretrained and Path(pretrained).exists():
        logger.info(f'Loading pretrained from: {pretrained}')
        model = load_model_weights(pretrained)
    elif pretrained and Path(str(pretrained).replace('/content/', f'{DRIVE_ROOT}/')).exists():
        prev = str(pretrained).replace('/content/', f'{DRIVE_ROOT}/')
        logger.info(f'Loading pretrained from Drive fallback: {prev}')
        model = load_model_weights(prev)

    dc = cfg['data']
    dataset = DocumentDataset(
        data_root='/content/tinydoc-vlm/data/synthetic',
        manifest_path=dc.get('manifest_path'),
        max_seq_length=dc.get('max_seq_length', 2048),
        stage=stage,
    )

    train_size = int(0.9 * len(dataset))
    eval_size = len(dataset) - train_size
    train_dataset, eval_dataset = random_split(dataset, [train_size, eval_size])
    logger.info(f'Dataset: {len(dataset)} total ({train_size} train / {eval_size} eval)')

    trainer_cfg = TrainerConfig(
        output_dir=str(local_dir),
        num_epochs=tc.get('num_epochs', 1),
        batch_size=tc.get('batch_size', 4),
        gradient_accumulation_steps=tc.get('gradient_accumulation_steps', 8),
        learning_rate=float(tc.get('learning_rate', 1e-4)),
        min_learning_rate=float(tc.get('min_learning_rate', 1e-5)),
        warmup_steps=100,
        weight_decay=tc.get('weight_decay', 0.01),
        max_grad_norm=tc.get('max_grad_norm', 1.0),
        max_seq_length=tc.get('max_seq_length', 2048),
        stage=stage,
        use_fp16=tc.get('use_fp16', True),
        save_every_steps=999999,
        eval_every_steps=tc.get('eval_every_steps', 250),
        log_every_steps=tc.get('log_every_steps', 10),
        gradient_checkpointing=tc.get('gradient_checkpointing', True),
        num_workers=1,
    )

    processor = TinyDocVLMProcessor()
    trainer = TinyDocVLMTrainer(
        model=model, processor=processor,
        train_dataset=train_dataset, eval_dataset=eval_dataset,
        config=trainer_cfg, device=device,
    )

    # Restore partial checkpoint from Drive (if local is gone but Drive has it)
    if not (local_dir / 'trainer_state.pt').exists() and (drive_dir / 'trainer_state.pt').exists():
        logger.info('Restoring partial checkpoint from Drive...')
        if local_dir.exists():
            shutil.rmtree(str(local_dir))
        shutil.copytree(str(drive_dir), str(local_dir))

    if (local_dir / 'trainer_state.pt').exists():
        ckpts = sorted(local_dir.glob('epoch_*'))
        if ckpts:
            trainer.load_checkpoint(str(ckpts[-1]))
            logger.info(f'Resumed from {ckpts[-1]}')

    trainer.train()

    # Mark complete
    (local_dir / 'stage_complete.txt').write_text('done')

    # Clean local: keep only best
    for d in local_dir.glob('init'):
        shutil.rmtree(d, ignore_errors=True)
    for d in sorted(local_dir.glob('step_*'))[:-1]:
        shutil.rmtree(d, ignore_errors=True)

    # Sync to Drive: overwrite old cleanly
    drive_dir.mkdir(parents=True, exist_ok=True)
    for item in drive_dir.iterdir():
        if item.is_dir():
            shutil.rmtree(str(item), ignore_errors=True)
        else:
            item.unlink()
    shutil.copytree(str(local_dir), str(drive_dir), dirs_exist_ok=True)
    logger.info(f'Stage {stage} synced to Drive')

for s in [1, 2, 3]:
    run_stage(s)

logger.info('\n===== ALL 3 STAGES COMPLETE! =====')
print('\nFinal model at:')
print(f'  {DRIVE_ROOT}/checkpoints/stage3/best')
print(f'  /content/checkpoints/stage3/best')

## 5. Download final model

In [ ]:
from google.colab import files
import os

DRIVE_ROOT = '/content/drive/MyDrive/tinydoc-vlm'

for path in [f'{DRIVE_ROOT}/checkpoints/stage3/best', '/content/checkpoints/stage3/best']:
    if os.path.exists(path):
        print(f'Found model at {path}')
        !zip -r /content/tinydoc-vlm-final.zip {path}
        files.download('/content/tinydoc-vlm-final.zip')
        break
else:
    print('Training not complete yet. Run cell 4 first.')